# Volatility Targeting: QQQ / BIL / GLD

This notebook demonstrates the `VolatilityTargeting` strategy on a three-asset universe:
- **QQQ** — Invesco QQQ Trust (Nasdaq-100 exposure)
- **BIL** — SPDR Bloomberg 1-3 Month T-Bill ETF (cash/short-duration)
- **GLD** — SPDR Gold Shares (defensive/inflation hedge)

**Inverse-volatility weighting** allocates more to low-volatility assets and less to high-volatility
assets each month, keeping the portfolio's realized volatility relatively stable over time.

An optional `target_vol` parameter de-levers the whole portfolio when realized vol exceeds the target.

**Rebalance**: Monthly mid-month (`month_mid` = 15th of each month)  
**Lookback**: 60 trading days for realized volatility estimate

In [1]:
from tiportfolio.helpers.cache import enable_data_source_cache
from tiportfolio.helpers.data import YFinance
from tiportfolio import (
    VolatilityTargeting, FixRatio, Schedule, ScheduleBasedEngine,
    compare_strategies, plot_strategy_comparison_interactive,
    rebalance_decisions_table,
)
import pandas as pd
import plotly.graph_objects as go

enable_data_source_cache("tiportfolio", cache_dir=".cache")

SYMBOLS   = ["QQQ", "BIL", "GLD"]
LOOKBACK  = 60
TARGET_VOL = None   # Set to e.g. 0.10 to cap annualized vol
START     = "2015-01-01"
END       = "2024-12-31"
INITIAL_VALUE = 10_000

print(f"Symbols: {SYMBOLS}  Lookback: {LOOKBACK}  TARGET_VOL: {TARGET_VOL}")

Symbols: ['QQQ', 'BIL', 'GLD']  Lookback: 60  TARGET_VOL: None


In [2]:
yf = YFinance(auto_adjust=True)
df = yf.query(SYMBOLS, start_date=START, end_date=END)

prices = {}
for symbol in df["symbol"].unique():
    sub = df[df["symbol"] == symbol].set_index("date")[["open", "high", "low", "close"]]
    prices[symbol] = sub

print(f"Loaded {len(prices)} symbols")
for sym, d in prices.items():
    print(f"  {sym}: {len(d)} rows  {d.index[0].date()} → {d.index[-1].date()}")

Loaded cached bar data.

Loaded 3 symbols
  BIL: 2515 rows  2015-01-02 → 2024-12-30
  GLD: 2515 rows  2015-01-02 → 2024-12-30
  QQQ: 2515 rows  2015-01-02 → 2024-12-30


In [3]:
strategy = VolatilityTargeting(
    symbols=SYMBOLS,
    lookback_days=LOOKBACK,
    target_vol=TARGET_VOL,
)

engine = ScheduleBasedEngine(
    allocation=strategy,
    rebalance=Schedule("quarter_mid"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)

result = engine.run(
    symbols=SYMBOLS,
    start=START, end=END,
    prices_df=prices,
)
print(result.summary())

Backtest Summary
----------------
Sharpe Ratio:        -1.2659
Sortino Ratio:       -1.5126
MAR Ratio:           0.6175
CAGR:                2.30%
Max Drawdown:        3.72%
Kelly Leverage:      -93.3693
Mean Excess Return:  -0.0172
Final Value:         12,546.91
Total Fee:           0.72
Rebalances:          40


/Users/tim/Development/TiPortfolio/src/tiportfolio/allocation/dynamic.py:47: UserWarning: VolatilityTargeting: insufficient history (1 rows < 61); using equal weights
  warnings.warn(
/Users/tim/Development/TiPortfolio/src/tiportfolio/allocation/dynamic.py:47: UserWarning: VolatilityTargeting: insufficient history (29 rows < 61); using equal weights
  warnings.warn(


In [4]:
fig = result.plot_portfolio(mark_rebalances=True)
fig.show()

## Weight Evolution Over Time

How the strategy allocates to QQQ, BIL, and GLD at each rebalance date.

In [5]:
weight_records = [
    {"date": d.date, **{s: d.target_weights.get(s, 0.0) for s in SYMBOLS}}
    for d in result.rebalance_decisions
]
weights_df = pd.DataFrame(weight_records).set_index("date")

fig_w = go.Figure()
colors = {"QQQ": "#636EFA", "BIL": "#00CC96", "GLD": "#FFA15A"}
for sym in SYMBOLS:
    fig_w.add_trace(go.Scatter(
        x=weights_df.index,
        y=weights_df[sym],
        name=sym,
        mode="lines+markers",
        line=dict(color=colors.get(sym)),
    ))
fig_w.update_layout(
    title="VolatilityTargeting: Weight Evolution per Rebalance",
    xaxis_title="Date",
    yaxis_title="Weight",
    yaxis_tickformat=".0%",
    legend_title="Symbol",
    hovermode="x unified",
)
fig_w.show()

In [6]:
decisions_df = rebalance_decisions_table(result)
decisions_df

,date,equity_before,equity_after,fee_paid,QQQ_price,QQQ_qty_before,QQQ_trade_qty,QQQ_qty_after,QQQ_value_after,BIL_price,BIL_qty_before,BIL_trade_qty,BIL_qty_after,BIL_value_after,GLD_price,GLD_qty_before,GLD_trade_qty,GLD_qty_after,GLD_value_after
0,2015-02-13 00:00:00+00:00,10242.507,10242.501,0.006,98.440,35.168,-0.485,34.683,3414.169,74.344,44.837,1.087,45.924,3414.169,117.98,29.219,-0.281,28.939,3414.169
1,2015-05-15 00:00:00+00:00,10321.309,10320.794,0.514,101.130,34.683,-32.533,2.150,217.423,74.311,45.924,87.254,133.178,9896.622,117.53,28.939,-27.175,1.763,207.263
2,2015-08-14 00:00:00+00:00,10300.494,10300.489,0.005,102.223,2.150,-0.161,1.989,203.349,74.279,133.178,-0.691,132.487,9840.995,106.85,1.763,0.634,2.397,256.150
3,2015-11-16 00:00:00+00:00,10295.140,10295.124,0.016,103.316,1.989,-1.136,0.853,88.108,74.279,132.487,2.663,135.150,10038.778,103.71,2.397,-0.775,1.622,168.253
4,2016-02-16 00:00:00+00:00,10302.276,10302.271,0.005,93.224,0.853,0.643,1.496,139.465,74.263,135.150,-0.817,134.333,9975.928,114.77,1.622,0.006,1.628,186.883
5,2016-05-16 00:00:00+00:00,10325.520,10325.518,0.003,99.648,1.496,0.352,1.848,184.117,74.279,134.333,0.101,134.434,9985.588,121.80,1.628,-0.349,1.279,155.816
6,2016-08-15 00:00:00+00:00,10356.912,10356.912,0.000,110.090,1.848,-0.033,1.815,199.814,74.311,134.434,-0.026,134.408,9988.044,127.84,1.279,0.043,1.322,169.054
7,2016-11-15 00:00:00+00:00,10345.084,10345.076,0.008,108.978,1.815,0.290,2.105,229.445,74.344,134.408,-1.424,132.984,9886.530,117.12,1.322,0.634,1.956,229.109
8,2017-02-15 00:00:00+00:00,10377.265,10377.255,0.011,121.592,2.105,0.858,2.963,360.290,74.381,132.984,-1.883,131.101,9751.458,117.45,1.956,0.304,2.261,265.517
9,2017-05-15 00:00:00+00:00,10415.855,10415.852,0.003,131.033,2.963,0.119,3.082,403.828,74.468,131.101,-0.532,130.569,9723.155,117.14,2.261,0.205,2.466,288.872


## Comparison to Baselines

- **Fixed 70/20/10**: Static allocation QQQ 70% / BIL 20% / GLD 10% — a naive starting point
- **Long QQQ only**: 100% QQQ buy-and-hold — maximum tech exposure, maximum drawdown

In [7]:
# Baseline 1: fixed 70/20/10
engine_fixed = ScheduleBasedEngine(
    allocation=FixRatio(weights={"QQQ": 0.70, "BIL": 0.20, "GLD": 0.10}),
    rebalance=Schedule("month_mid"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)
result_fixed = engine_fixed.run(
    symbols=SYMBOLS, start=START, end=END, prices_df=prices
)
print("Fixed 70/20/10")
print(result_fixed.summary())

Fixed 70/20/10
Backtest Summary
----------------
Sharpe Ratio:        0.6928
Sortino Ratio:       0.8879
MAR Ratio:           0.5442
CAGR:                14.34%
Max Drawdown:        26.35%
Kelly Leverage:      4.5278
Mean Excess Return:  0.1060
Final Value:         38,159.55
Total Fee:           1.22
Rebalances:          120


In [8]:
# Baseline 2: 100% QQQ
engine_qqq = ScheduleBasedEngine(
    allocation=FixRatio(weights={"QQQ": 1.0}),
    rebalance=Schedule("month_mid"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)
result_qqq = engine_qqq.run(
    symbols=["QQQ"], start=START, end=END, prices_df={"QQQ": prices["QQQ"]}
)
print("Long QQQ only")
print(result_qqq.summary())

Long QQQ only
Backtest Summary
----------------
Sharpe Ratio:        0.7021
Sortino Ratio:       0.8942
MAR Ratio:           0.5242
CAGR:                18.41%
Max Drawdown:        35.12%
Kelly Leverage:      3.2174
Mean Excess Return:  0.1532
Final Value:         54,123.67
Total Fee:           0.00
Rebalances:          0


In [9]:
compare_strategies(
    result, result_fixed, result_qqq,
    names=["VolatilityTargeting", "Fixed 70/20/10", "Long QQQ"],
)

,VolatilityTargeting,Fixed 70/20/10,Long QQQ,better
metric,,,,
sharpe_ratio,-1.265934,0.692778,0.702109,Long QQQ
sortino_ratio,-1.512634,0.887859,0.894231,Long QQQ
mar_ratio,0.617546,0.544195,0.524226,VolatilityTargeting
cagr,0.022964,0.143405,0.184102,Long QQQ
max_drawdown,0.037186,0.263518,0.351187,VolatilityTargeting


In [10]:
plot_strategy_comparison_interactive(
    result, result_fixed, result_qqq,
)

## Target Vol Demo: De-levering Effect

Re-run with `TARGET_VOL=0.10` (10% annualized). When portfolio volatility exceeds this threshold,
all weights are scaled down proportionally, effectively moving capital to cash (BIL gets higher relative weight).

In [11]:
strategy_capped = VolatilityTargeting(
    symbols=SYMBOLS,
    lookback_days=LOOKBACK,
    target_vol=0.10,
)

engine_capped = ScheduleBasedEngine(
    allocation=strategy_capped,
    rebalance=Schedule("month_mid"),
    fee_per_share=0.0035,
    initial_value=INITIAL_VALUE,
)

result_capped = engine_capped.run(
    symbols=SYMBOLS, start=START, end=END, prices_df=prices
)
print("VolatilityTargeting target_vol=0.10")
print(result_capped.summary())

VolatilityTargeting target_vol=0.10
Backtest Summary
----------------
Sharpe Ratio:        -1.5310
Sortino Ratio:       -1.8585
MAR Ratio:           0.6128
CAGR:                2.20%
Max Drawdown:        3.59%
Kelly Leverage:      -129.1186
Mean Excess Return:  -0.0182
Final Value:         12,426.22
Total Fee:           0.88
Rebalances:          120


/Users/tim/Development/TiPortfolio/src/tiportfolio/allocation/dynamic.py:47: UserWarning:

VolatilityTargeting: insufficient history (1 rows < 61); using equal weights

/Users/tim/Development/TiPortfolio/src/tiportfolio/allocation/dynamic.py:47: UserWarning:

VolatilityTargeting: insufficient history (9 rows < 61); using equal weights

/Users/tim/Development/TiPortfolio/src/tiportfolio/allocation/dynamic.py:47: UserWarning:

VolatilityTargeting: insufficient history (29 rows < 61); using equal weights

/Users/tim/Development/TiPortfolio/src/tiportfolio/allocation/dynamic.py:47: UserWarning:

VolatilityTargeting: insufficient history (49 rows < 61); using equal weights



In [12]:
compare_strategies(
    result, result_capped,
    names=["Uncapped (no target_vol)", "Capped (target_vol=0.10)"],
)

,Uncapped (no target_vol),Capped (target_vol=0.10),better
metric,,,
sharpe_ratio,-1.265934,-1.531043,Uncapped (no target_vol)
sortino_ratio,-1.512634,-1.858529,Uncapped (no target_vol)
mar_ratio,0.617546,0.612830,Uncapped (no target_vol)
cagr,0.022964,0.021975,Uncapped (no target_vol)
max_drawdown,0.037186,0.035859,Capped (target_vol=0.10)


In [13]:
plot_strategy_comparison_interactive(result, result_capped)

## Portfolio Beta

Track portfolio beta over time vs SPY.


In [14]:
# Plot portfolio beta over time
fig_beta = result.plot_portfolio_beta()
fig_beta.show()


Loaded cached bar data.



## Interpretation

### Inverse-Volatility Weighting
At each rebalance, each asset's weight is proportional to `1 / σ_i` (inverse annualized volatility
over the lookback window), normalized so weights sum to 1.0:

$$w_i = \frac{1/\sigma_i}{\sum_j 1/\sigma_j}$$

**Effect**: When QQQ becomes more volatile (e.g. during market stress), its weight decreases
automatically; BIL and GLD — typically lower-volatility — receive larger shares. This keeps
the portfolio's realized volatility more stable than a static allocation.

### Monthly Mid-Month Rebalance
Rebalancing on the 15th of each month limits turnover. Between rebalances, weights drift with
market prices — this acts as a natural "freezing period" preventing reaction to daily noise.

### Target Vol Mechanics
When `target_vol` is set, a scalar `s = min(target_vol / portfolio_vol, 1.0)` is applied to all
weights after normalization. If `portfolio_vol < target_vol`, no de-levering occurs (s=1). Only
when portfolio vol exceeds the target does the strategy scale back, shifting effective weight toward
cash (BIL already included in the universe).

### QQQ/BIL/GLD Synergy
- **QQQ**: high-growth, high-vol → receives lower weight in turbulent markets
- **BIL**: near-zero vol → acts as a shock absorber, weight increases when equities spike
- **GLD**: moderate vol, often negatively correlated to equities in crises → provides diversification

> **Note**: Past volatility predicts future volatility better than future returns. Inverse-vol
> weighting is a risk-parity concept, not a return-prediction model.